In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import os
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
print("Loading data and mapping files...")
meta_df = pd.read_parquet('../data/clean_meta.parquet')

with open('../data/processed/item_mapping.json', 'r') as f:
    item_mapping = json.load(f)

with open('../data/processed/user_mapping.json', 'r') as f:
    user_mapping = json.load(f)

user_item_edge_index = torch.load('../data/processed/user_item_edge_index.pt')

num_items = len(item_mapping)
num_users = len(user_mapping)
hidden_dim = 384 # Output dimension of all-MiniLM-L6-v2

print(f"Items to process: {num_items}")
print(f"Users to process: {num_users}")

In [ ]:
# 1. Prepare texts for all items in order of their mapped ID (0 to num_items-1)
# Create a reverse mapping (ID -> text)
item_texts = [""] * num_items

# We iterate over meta_df to map ASIN to Text, then place it in the correct ID slot
for idx, row in meta_df.iterrows():
    asin = row['asin']
    if asin in item_mapping:
        mapped_id = item_mapping[asin]
        
        # safely handle title
        title = row['title']
        if pd.isna(title):
            title = ""
            
        # safely handle description which might be a list or missing
        desc = row['description']
        desc_str = ""
        if isinstance(desc, list):
            desc_str = " ".join([str(x) for x in desc])
        elif pd.notna(desc):
            desc_str = str(desc)
            
        full_text = f"{title}. {desc_str}".strip()
        item_texts[mapped_id] = full_text

In [ ]:
# 2. Encode Item Features
print("Loading sentence-transformer model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding text... This may take a few moments.")
# batch size can be customized, 32 or 64 is typical
item_features_np = model.encode(item_texts, show_progress_bar=True, batch_size=128)
item_features = torch.tensor(item_features_np, dtype=torch.float)

print(f"Successfully generated item features of shape: {item_features.shape}")

In [ ]:
# 3. Initialize User Features
print("Generating user features based on historical interaction means...")
user_features = torch.zeros((num_users, hidden_dim), dtype=torch.float)

u_nodes, i_nodes = user_item_edge_index

# Convert to pandas for quick grouping by user (numpy operations are also fine)
# For each user, get all item indices they interacted with
u_array = u_nodes.numpy()
i_array = i_nodes.numpy()

# We can use pd Series grouping or a simple loop depending on speed
from collections import defaultdict
user_to_items = defaultdict(list)
for u, i in zip(u_array, i_array):
    user_to_items[u].append(i)

# Calculate mean feature vectors for users
for u in range(num_users):
    items_interacted = user_to_items.get(u, [])
    if len(items_interacted) > 0:
        # get the embeddings for these items
        interacted_embs = item_features[items_interacted] # shape: [num_interactions, hidden_dim]
        mean_emb = torch.mean(interacted_embs, dim=0)
        user_features[u] = mean_emb
    else:
        # If no interactions (e.g., they only interacted with items outside the dataset or missing valid edges)
        # leave as zeros
        pass

print(f"Successfully generated user features of shape: {user_features.shape}")

In [ ]:
# 4. Save to disk
torch.save(item_features, '../data/processed/x_dict_item.pt')
torch.save(user_features, '../data/processed/x_dict_user.pt')
print("Encoding complete! Artifacts saved to data/processed/")